# Assignment 1 – Machine Learning Foundations: Data Preparation

**GitHub Repository:** https://github.com/YOUR_USERNAME/ML-fundamentals-2026

---

This notebook implements all data preparation and feature engineering tasks for the UCI Bank Marketing dataset (`bank-additional.csv`).  
The emphasis is on **correct pipeline ordering**, **principled decision-making**, and **avoidance of data leakage** — not on maximising model accuracy.  
All tasks are addressed in a methodologically sound sequence that is justified in the **Task Ordering** section.


In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.utils import resample

RANDOM_STATE = 42
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
sns.set_theme(style='whitegrid')
print('✓ All imports successful')

---
## Task Ordering
*Lecture 2 – Data Splitting and Leakage | Lecture 5 – Preprocessing | Lecture 9 – ML Pipeline*

The assignment lists tasks alphabetically. A correct ML pipeline demands a specific execution order to prevent **data leakage** — the inadvertent use of information from validation or test splits during training-side transformations.

### Chosen Execution Order

| Step | Task | Information allowed | Information forbidden | Leakage if order violated |
|:----:|------|---------------------|-----------------------|---------------------------|
| 1 | Identifying the Prediction Target | Full dataset (read-only) | None | None |
| 2 | Data Loading & Exploration | Full dataset (read-only) | Fitting any statistical transformer | Preprocessing leakage |
| 3 | Managing Missing Values *(structural cleaning)* | Full dataset | Statistical estimation (means, medians) | None — no parameters fitted |
| **4** | **Data Splitting** | Full dataset | — | Everything downstream becomes leaky |
| 5 | Managing Missing Values *(imputation)* | Training set only | Val/test statistics | Imputed values reflect test distribution |
| 6 | Encoding Categorical Variables | Training vocabulary only | Val/test category counts | Encoder vocabulary biased by test categories |
| 7 | Feature Scaling | Training mean/std only | Val/test mean or std | Scaler mean/std biased by test distribution |
| 8 | Feature Selection | Training variances/correlations only | Val/test distributions | Feature set tuned to test data |
| 9 | Addressing Class Imbalance | Training set only | Val/test samples | Synthetic samples duplicate test observations |
| 10 | Training a Logistic Regression Model | Resampled training set | Val/test labels | Direct label leakage |

### What Information Is Allowed / Forbidden at Each Stage

- **Steps 1–3 (pre-split):** May use the whole dataset for inspection and structural cleaning. Must not fit any statistical transformer.
- **Step 4 (split):** Partitions the data; defines the boundary between what the model may learn from and what must remain unseen.
- **Steps 5–9 (post-split, training set only):** All parameter estimation (means, standard deviations, encoder mappings, resampling) must be derived exclusively from the training partition.
- **Step 10 (evaluation):** Predictions are made on validation data that has only been *transformed* (never fit upon). The test set remains untouched throughout; it is reserved for a final unbiased evaluation after all pipeline decisions are finalised. Using it for any intermediate check would allow those decisions to be implicitly tuned to test performance, reintroducing leakage through the back door.

### Examples of Incorrect Ordering and Their Consequences

**Scaling before splitting:** If `StandardScaler` is fitted on the entire dataset, the computed mean and standard deviation incorporate information from validation and test observations. When those sets are subsequently scaled using these statistics, the model indirectly benefits from having seen the test distribution — a form of *preprocessing leakage*. The practical consequence is that reported validation accuracy and recall are optimistically biased: the model appears to generalise better than it would on truly unseen data, potentially by several percentage points when features have high variance.

**Imputation before splitting:** If median imputation is fitted on the full dataset, the imputed value reflects the combined distribution of training and test observations. Downstream transformers and the model itself indirectly benefit from test-set statistics in every imputed cell — a subtler but equally real form of leakage.

**Resampling before splitting:** If synthetic or duplicated minority samples are created from the full dataset before splitting, duplicates of the same original observation can appear in both the training and validation sets. The model is then evaluated on data it has already seen, producing inflated recall and precision.

---
## Step 1 – Identifying the Prediction Target
*Lecture 1 – Problem Formulation | Lecture 2 – Data Inspection*

### Correct Target Variable

The target variable is **`y`** — a binary indicator of whether the client subscribed to a term deposit (`yes`) or not (`no`). This column directly corresponds to the campaign objective stated in the problem description: *predict whether a client subscribes to a term deposit given information available at the time of contact*. It is the only column that represents the actual business outcome the bank wants to automate.

### Variables That Should NOT Be Treated as the Target

1. **`duration`** (last contact call duration, in seconds): This is a case of **temporal leakage** (Lecture 1 — Problem Formulation): the variable is generated *after* the event we are trying to predict, making it unavailable at inference time. A deployed model must decide who to call *before* the call is made, so `duration` is simply not observable at prediction time. A model trained with `duration` as a feature would achieve unrealistically high performance and be useless in production.

2. **`campaign`** (number of contacts performed during this campaign): This variable is **endogenous** with respect to the outcome — agents tend to call unresponsive clients more times, so `campaign` is partly a *consequence* of the client's eventual decision. Using it as a target conflates cause and effect. It also accumulates across the campaign, making it a time-varying quantity that cannot serve as a static prediction objective.

3. **`pdays`** (days since the client was last contacted from a prior campaign): This records a scheduling artefact, not a campaign outcome. The sentinel value 999 means 'never contacted', blending a categorical meaning into a numerical column. It cannot serve as a meaningful prediction target. Note that `pdays` is retained as an *engineered feature* (binary indicator of prior contact) because it carries predictive signal about client receptiveness — being previously contacted vs. never contacted is qualitatively different from the mere count of days elapsed.

---
## Step 2 – Data Loading and Exploration
*Lecture 1 – Problem Formulation | Lecture 2 – Data Inspection and EDA*

In [ ]:
# Load the dataset (semicolon-separated as provided by UCI)
df_raw = pd.read_csv('bank-additional.csv', sep=';')

print(f'Dataset shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Memory usage  : {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB')
df_raw.head()

In [ ]:
# ── Column types ──────────────────────────────────────────────────────────────
num_cols_raw = df_raw.select_dtypes(include='number').columns.tolist()
cat_cols_raw = [c for c in df_raw.columns if c not in num_cols_raw]

print('NUMERICAL variables:', num_cols_raw)
print()
print('CATEGORICAL variables:', cat_cols_raw)

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
df_raw[num_cols_raw].describe().round(3)

In [ ]:
# ── Target distribution ────────────────────────────────────────────────────────
target_vc = df_raw['y'].value_counts()
target_pct = df_raw['y'].value_counts(normalize=True).mul(100).round(1)
print('Target variable distribution:')
print(pd.DataFrame({'Count': target_vc, 'Percentage (%)': target_pct}))
print(f'\nImbalance ratio (majority:minority) ≈ {target_vc["no"]/target_vc["yes"]:.1f}:1')

**Observation:** The dataset is **heavily imbalanced** — roughly 89% of clients did not subscribe (`no`) and only ~11% did (`yes`). This is a ~8:1 imbalance ratio, which will affect model training (the classifier will be biased towards the majority class) and evaluation (accuracy alone will be a misleading metric). This issue is addressed in the Class Imbalance task.

In [ ]:
# ── Explicit missing values ────────────────────────────────────────────────────
explicit_na = df_raw.isnull().sum()
print('Explicit NaN values per column:')
print(explicit_na[explicit_na > 0] if explicit_na.sum() > 0 else '  None found.')

# ── Implicit missing values: 'unknown' category ────────────────────────────────
print('\nImplicit missing values ("unknown" category):')
for col in cat_cols_raw:
    n = (df_raw[col] == 'unknown').sum()
    if n > 0:
        print(f'  {col:<15}: {n:>4} ({100*n/len(df_raw):.1f}%)')

# ── Sentinel numerical value: pdays == 999 ─────────────────────────────────────
n_999 = (df_raw['pdays'] == 999).sum()
print(f'\nSentinel value pdays == 999: {n_999} ({100*n_999/len(df_raw):.1f}%)  → means "never contacted before"')

**Findings:**
- **No explicit NaN values** exist in this dataset.
- **Implicit missingness** appears in several categorical variables via the `'unknown'` label. The most affected column is `default` (19.5%), suggesting many clients declined to disclose their credit default status. This is a potentially informative form of missingness: clients who withhold sensitive financial information may behave differently from those who disclose it. Retaining `'unknown'` as a distinct category (rather than imputing or dropping) is therefore a deliberate modelling decision, not merely a convenience — it allows the model to learn a separate coefficient for this group.
- **`pdays`** uses the sentinel value `999` to mean 'client was never contacted in a previous campaign'. This is not a genuine numerical measurement and must be treated separately from the real duration values.

In [ ]:
# ── Visualisation 1: Distribution of two numerical variables ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Age
axes[0].hist(df_raw['age'], bins=30, color='#2196F3', edgecolor='white', linewidth=0.6)
axes[0].set_title('Distribution of Age', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')
axes[0].axvline(df_raw['age'].median(), color='#F44336', linestyle='--', linewidth=1.5, label=f'Median = {df_raw["age"].median():.0f}')
axes[0].legend()

# Euribor 3-month rate
axes[1].hist(df_raw['euribor3m'], bins=30, color='#FF9800', edgecolor='white', linewidth=0.6)
axes[1].set_title('Distribution of Euribor 3-Month Rate', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Euribor 3m (%)')
axes[1].set_ylabel('Count')
axes[1].axvline(df_raw['euribor3m'].median(), color='#F44336', linestyle='--', linewidth=1.5,
                label=f'Median = {df_raw["euribor3m"].median():.2f}')
axes[1].legend()

plt.suptitle('Numerical Variable Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Age: Roughly bell-shaped with a slight right skew; most clients are 25–60 years old.')
print('Euribor3m: Strongly bimodal — two dominant interest-rate regimes present in the data.')
print('  This bimodality reflects different macroeconomic periods captured in the campaign data.')

In [ ]:
# ── Visualisation 2: Distribution of two categorical variables ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Job
job_vc = df_raw['job'].value_counts()
colors_job = ['#EF5350' if v == 'unknown' else '#42A5F5' for v in job_vc.index]
axes[0].barh(job_vc.index[::-1], job_vc.values[::-1], color=colors_job[::-1])
axes[0].set_title('Job Distribution', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Count')
red_patch = mpatches.Patch(color='#EF5350', label='unknown')
blue_patch = mpatches.Patch(color='#42A5F5', label='known')
axes[0].legend(handles=[blue_patch, red_patch], fontsize=9)

# Education
edu_vc = df_raw['education'].value_counts()
colors_edu = ['#EF5350' if v == 'unknown' else '#66BB6A' for v in edu_vc.index]
axes[1].barh(edu_vc.index[::-1], edu_vc.values[::-1], color=colors_edu[::-1])
axes[1].set_title('Education Distribution', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Count')
red_patch2 = mpatches.Patch(color='#EF5350', label='unknown')
green_patch = mpatches.Patch(color='#66BB6A', label='known')
axes[1].legend(handles=[green_patch, red_patch2], fontsize=9)

plt.suptitle('Categorical Variable Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Job: Dominated by admin., blue-collar, and technician. 39 "unknown" values (red).')
print('Education: University degree is the most common. 167 "unknown" values (red) — 4% of rows.')

### Variables Requiring Special Consideration

1. **`duration`** – *Not available at prediction time.* The call duration is recorded only after the conversation ends. A deployed model must decide who to call *before* the call is made, so `duration` is unavailable at inference. This is a temporal leakage problem: including it would make the model unrealistically accurate and useless in production. **Decision: drop before any modelling.**

2. **`pdays`** – *Sentinel encoding mixed with numerical values.* 96% of entries are `999`, meaning the client was never previously contacted. This is not a continuous measurement; it is a categorical flag embedded in a numerical column. Using it as a raw numeric feature would mislead a linear model into treating 999 as an extreme continuous value on the same scale as observations like 3 or 12. **Decision: replace 999 with NaN; create a binary indicator `pdays_contacted`.**

3. **`emp.var.rate`, `euribor3m`, `nr.employed`** – *Probable multicollinearity.* These three macroeconomic indicators all capture the same underlying economic cycle — employment conditions, interest rates, and labour market size tend to move together. Based on domain knowledge, they are likely to be highly correlated; this will be confirmed by computing the correlation matrix on the training set in Feature Selection, where any redundant features will be removed. Retaining all three would inflate coefficient variance in Logistic Regression and make the L2 regularisation penalty less effective. Note: `euribor3m` is visualised above to illustrate its bimodal distribution — this pattern reflects two distinct macroeconomic regimes in the data and anticipates the multicollinearity finding addressed in Feature Selection.

---
## Step 3 – Managing Missing Values (Pre-Split Structural Cleaning)
*Lecture 2 – Data Inspection | Lecture 5 – Preprocessing and Pipeline Discipline*

We distinguish two categories of decisions, following the lecture distinction between **stateless** and **stateful** transformations (Lecture 5):

- **Stateless (data cleaning) decisions** — safe to apply before the split: structural choices that require no parameter estimation from the data, such as flagging a sentinel value, dropping a column for conceptual reasons, or re-encoding the target. These operations are deterministic and cannot introduce leakage because they learn nothing from the data distribution.
- **Stateful (modelling) decisions** — must be applied after the split, fitted on training only: any operation that estimates a statistic from the data, such as computing a median for imputation, must be fitted exclusively on the training set. Applying a training-fitted value to validation and test sets ensures those splits never influence the imputed value.

### Pre-Split Actions (Stateless — No Parameters Estimated)

In [ ]:
# Work on a copy to preserve the raw dataframe for reference
df = df_raw.copy()

# ── 1. Drop 'duration': unavailable at prediction time ────────────────────────
df.drop(columns=['duration'], inplace=True)
print('✓ Dropped "duration" — not available at prediction time (see EDA discussion).')

# ── 2. Create binary indicator for pdays sentinel ─────────────────────────────
# Semantic content: was the client ever contacted in a previous campaign?
# This binary distinction (ever-contacted vs. never-contacted) carries predictive signal.
df['pdays_contacted'] = (df['pdays'] != 999).astype(int)
# Replace sentinel 999 with NaN to represent genuine missingness for imputation later
df['pdays'] = df['pdays'].replace(999, np.nan)
print('✓ Created "pdays_contacted" indicator (1 = was contacted before, 0 = never).')
print('✓ Replaced pdays sentinel 999 → NaN.')

# ── 3. Encode target as binary integer ────────────────────────────────────────
df['y'] = (df['y'] == 'yes').astype(int)
print('✓ Target encoded: yes → 1, no → 0.')

print(f'\nDataset shape after pre-split cleaning: {df.shape}')

### Post-Split Missingness Strategy (Deferred to After Splitting)

| Variable | Type of Missingness | Strategy | Fit on training? |
|----------|---------------------|----------|------------------|
| `pdays` | Sentinel 999 → NaN (~4% genuine values) | Median imputation | ✓ Yes — median computed on training set only |
| `job`, `marital`, `education`, `housing`, `loan` | Implicit `'unknown'` category | Keep as a distinct category | No — this is a stateless decision: no statistic is estimated; the encoder vocabulary simply includes `'unknown'` as a valid level |
| `default` | 19.5% `'unknown'` — largest affected column | Keep as distinct category; near-constant `yes` class deferred to Feature Selection | No — structural decision; low variance is confirmed on the training set in Step 8 |

**Why not drop rows with `unknown`?** Dropping all rows containing `'unknown'` in any categorical column would remove up to ~24% of the training data. Beyond the loss of statistical power, it would discard the potentially informative signal that non-disclosure carries. Retaining `'unknown'` as a distinct category is parameter-free: the one-hot encoder simply adds a binary column for it, no statistic is estimated, and no leakage can occur.

**A note on `default`:** EDA on the full dataset reveals that `default = 'yes'` is extremely rare (3 observations out of 4,119). This observation motivates checking `default_yes` for near-zero variance in Feature Selection — but the formal drop decision is made on training-set variance only, not on this full-dataset count.

---
## Step 4 – Data Splitting
*Lecture 2 – Data Splitting and Leakage | Lecture 9 – ML Pipeline*

### Split Proportions: 70 / 15 / 15

With 4,119 observations:
- **Training (70% ≈ 2,883 rows):** Sufficient for Logistic Regression to converge and for stateful transformations (scaler, imputer, encoder) to estimate stable parameters.
- **Validation (15% ≈ 618 rows):** Large enough to produce reliable intermediate performance estimates for pipeline coherence checking.
- **Test (15% ≈ 618 rows):** Held out entirely and never used for any decision in this notebook. Its sole purpose is to provide a final unbiased performance estimate once all pipeline choices are fixed. Inspecting it at any earlier stage — even to check class distribution — would allow pipeline decisions to be implicitly tuned to its properties, reintroducing leakage.

### Why Stratification Is Necessary

The target is imbalanced (~11% positive). Without stratification, a random split could produce a validation or test set with a materially different class ratio (e.g., 7% or 15% positives), making cross-split performance comparisons unreliable. `stratify=y` ensures all three partitions preserve the original ~89/11 distribution.

As established in the Task Ordering section, all stateful transformations must be fitted exclusively on the training partition. Performing the split later would introduce preprocessing leakage and inflate validation metrics.

In [ ]:
X = df.drop(columns=['y'])
y_series = df['y']

# Step 1: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_series, test_size=0.30, random_state=RANDOM_STATE, stratify=y_series
)

# Step 2: split temporary 50/50 → validation and test (each 15% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

# Reset indices for clean downstream operations
for split in [X_train, X_val, X_test, y_train, y_val, y_test]:
    split.reset_index(drop=True, inplace=True)

print(f'{"Split":<12} {"Rows":>6} {"Positive %":>12}')
print('-' * 32)
for name, Xs, ys in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    print(f'{name:<12} {len(Xs):>6,} {ys.mean()*100:>11.1f}%')

The class ratio is preserved across all three splits (≈11% positive in each), confirming that stratification worked correctly.

---
## Step 5 – Managing Missing Values (Post-Split Imputation)
*Lecture 2 – Data Inspection | Lecture 5 – Preprocessing and Pipeline Discipline*

In [ ]:
# ── Impute pdays with TRAINING-SET median ─────────────────────────────────────
# Rationale: pdays is right-skewed (values range from 1 to ~27 for the ~4% of
# actual observations). The median is more robust to skew than the mean.
# CRITICAL: the median is computed only on X_train to avoid leaking test statistics.

pdays_median = X_train['pdays'].median()
print(f'Training-set median of pdays (non-999 values): {pdays_median}')

X_train['pdays'] = X_train['pdays'].fillna(pdays_median)
X_val['pdays']   = X_val['pdays'].fillna(pdays_median)
X_test['pdays']  = X_test['pdays'].fillna(pdays_median)

print('✓ pdays NaN imputed with training-set median in all splits.')
print(f'  Remaining NaN in pdays: train={X_train["pdays"].isnull().sum()}, '
      f'val={X_val["pdays"].isnull().sum()}, test={X_test["pdays"].isnull().sum()}')

**Why the median?** The non-sentinel `pdays` values represent days since the last contact, which is right-skewed (most recent contacts cluster in the low single digits; a few extend to several weeks). The median is more robust to this skew than the mean, providing a more representative central imputation value.

**Why fitted on training data only?** If the median were computed on the full dataset, it would incorporate the distribution of validation and test observations. All downstream transformers and the model itself would then indirectly benefit from having seen the test distribution — a stateful leakage that makes evaluation metrics optimistically biased.

---
## Step 6 – Encoding Categorical Variables
*Lecture 4 – Categorical Encoding | Lecture 6 – Linear Models | Lecture 9 – Feature Engineering*

### Identifying Categorical Variables

In [ ]:
# Detect categorical columns robustly across pandas versions
cat_features = []
for c in X_train.columns:
    try:
        X_train[c].str.lower()
        cat_features.append(c)
    except AttributeError:
        pass
print('Categorical features to encode:', cat_features)

### Nominal vs. Ordinal Classification

| Variable | Type | Justification |
|----------|------|---------------|
| `job` | **Nominal** | No inherent ordering among professions. Assigning integers would imply `admin. > blue-collar` in the linear model, which is meaningless. |
| `marital` | **Nominal** | No meaningful rank among single, married, divorced. |
| `contact` | **Nominal** | Contact medium (cellular vs. telephone) — two unordered categories. |
| `month` | **Nominal** | Treating `month` as ordinal would assign consecutive integers (e.g., Jan=1 … Dec=12), implying that December is *more* of something than November. Logistic Regression's linear coefficient would attempt to exploit this ordering, producing a spurious monotone relationship between month-integer and log-odds. Since months here label distinct campaign periods with no intrinsic rank, one-hot encoding is correct. |
| `day_of_week` | **Nominal** | Same reasoning as `month` — days encode scheduling patterns, not a true ordinal. |
| `poutcome` | **Nominal** | Previous outcome (failure / nonexistent / success) — no cardinal ordering; a linear coefficient over integers would be arbitrary. |
| `default` | **Nominal** | Credit default status (no / unknown / yes) — unordered categorical. |
| `housing` | **Nominal** | Has housing loan (yes / no / unknown). |
| `loan` | **Nominal** | Has personal loan (yes / no / unknown). |
| `education` | **Ordinal** | Clear, domain-justified progression: illiterate < basic.4y < basic.6y < basic.9y < high.school < professional.course < university.degree. Ordinal integer encoding is appropriate here: a single coefficient captures the general trend that higher education correlates with financial product adoption, without inflating dimensionality. |

### Encoding Strategy

- **Nominal variables → One-Hot Encoding (OHE):** OHE creates a binary column for each category level. For Logistic Regression, assigning arbitrary integers to nominal categories (e.g., admin.=1, blue-collar=2) would imply a false magnitude ordering — the model's linear coefficient would treat higher integers as more subscription-predictive, which is semantically wrong.
- **Ordinal variable (`education`) → Integer mapping:** The natural order is mapped directly to integers 0–6. The model learns a single coefficient capturing the directional trend, without inflating the feature space by $k-1$ columns. See the note below on the `'unknown'` level.
- **Dropping one OHE column per variable (`drop_first=True`):** With $k$ OHE columns for a $k$-level variable, the last column is perfectly predicted by the others (they sum to 1). This **dummy variable trap** produces perfect multicollinearity, making the design matrix rank-deficient and destabilising Logistic Regression's gradient-based optimisation. Dropping one column resolves this.
- **All encoders fitted on training data only:** The OHE vocabulary (the set of valid category levels) is derived from the training set. Validation and test sets are aligned to this schema via `reindex`, ensuring the model never encounters categories absent from training.

**Note on `education = 'unknown'`:** Assigning `unknown` integer value 7 — higher than `university.degree` (6) — would imply to the linear model that unknown education is associated with *more* education than a degree, which is semantically incorrect. We acknowledge this limitation: `unknown` is placed at the end as a pragmatic choice, and its coefficient should be interpreted as an intercept adjustment for the non-disclosure group rather than as an education level. An alternative would be mode imputation of `unknown` values from the training set before encoding; we retain the current approach for transparency and note the caveat.

In [ ]:
# ── Ordinal encoding for 'education' ──────────────────────────────────────────
education_order = [
    'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
    'high.school', 'professional.course', 'university.degree', 'unknown'
]
# 'unknown' is placed at the end — it will receive the highest integer,
# but this is acceptable because 'unknown' carries its own meaning.
# Alternatively we could impute it with the mode; we keep it explicit.
edu_map = {cat: i for i, cat in enumerate(education_order)}
print('Education ordinal mapping:', edu_map)

for split in [X_train, X_val, X_test]:
    split['education'] = split['education'].map(edu_map)

print('✓ Ordinal encoding applied to "education".')

In [ ]:
# ── One-Hot Encoding for all remaining nominal categorical variables ───────────
nominal_cols = [c for c in cat_features if c != 'education']
print('Nominal columns to OHE:', nominal_cols)

# Fit OHE vocabulary on TRAINING set only
# We use pd.get_dummies logic manually to control drop_first and handle unseen categories
X_train_enc = pd.get_dummies(X_train, columns=nominal_cols, drop_first=True, dtype=int)

# Apply the SAME columns to validation and test, filling any missing columns with 0
X_val_enc  = pd.get_dummies(X_val,  columns=nominal_cols, drop_first=True, dtype=int)
X_test_enc = pd.get_dummies(X_test, columns=nominal_cols, drop_first=True, dtype=int)

# Align to training columns (add missing cols as 0, drop extra cols not in train)
train_cols = X_train_enc.columns.tolist()
X_val_enc  = X_val_enc.reindex(columns=train_cols, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=train_cols, fill_value=0)

print(f'\nDimensionality before encoding : {X_train.shape[1]} features')
print(f'Dimensionality after  encoding : {X_train_enc.shape[1]} features')
print(f'New columns added              : {X_train_enc.shape[1] - X_train.shape[1]}')

### Effect of Encoding on Dimensionality, Interpretability, and Decision Boundaries

- **Dimensionality:** OHE expands each nominal variable with $k$ levels into $k-1$ binary columns (after `drop_first`). The feature space grows from 20 to 47 columns. This increases expressiveness at the cost of sparsity — most OHE columns will be 0 for any given observation.

- **Interpretability of coefficients:** Each OHE column's coefficient directly represents the log-odds contribution of belonging to that category *relative to the dropped reference category*. For example, if `poutcome_success` has coefficient +1.2, a client with a successful prior outcome has 1.2 higher log-odds of subscribing than a client with a failed prior outcome (the reference). This is directly interpretable under the Logistic Regression model.

- **Decision boundaries:** Logistic Regression's decision boundary is a hyperplane defined by $w^Tx + b = 0$. Without encoding, a single integer coefficient for `job` would impose a monotone relationship between job-integer and log-odds — meaningless for a nominal variable. With OHE, each category receives its own coefficient, allowing the hyperplane to shift independently for each job type. This is equivalent to fitting a separate intercept per category — the maximum expressiveness a *linear* model can achieve on nominal categorical inputs.

---
## Step 7 – Feature Scaling
*Lecture 5 – Feature Scaling | Lecture 6 – Logistic Regression and Optimisation*

### Variables That Require Scaling

All **continuous numerical variables** require scaling. Binary indicator variables (OHE columns, `pdays_contacted`) already live on a [0,1] scale and do not need scaling. The ordinal-encoded `education` variable (range 0–6) is included in scaling to put it on a comparable scale with the other continuous features.

### Scaling Method: Standardisation (Z-score normalisation)

**StandardScaler** transforms each feature to zero mean and unit variance: $z = (x - \mu) / \sigma$.

**Justification for Logistic Regression:**

1. **Gradient-based optimisation:** Logistic Regression minimises the log-loss via L-BFGS (a quasi-Newton method). The partial derivative of the loss with respect to each coefficient is proportional to the magnitude of the corresponding feature. When features have vastly different scales — e.g., `age` ∈ [18, 95] vs. `cons.price.idx` ∈ [92.2, 94.8] — the gradient components along large-scale dimensions dominate, while those along small-scale dimensions are negligible. The resulting loss landscape is elongated and ill-conditioned: the optimiser must take many small steps to traverse the narrow dimensions, causing slow or oscillating convergence. Standardisation rescales all partial derivatives to a comparable magnitude, making the landscape more spherical and allowing the optimiser to converge in fewer iterations.

2. **Coefficient comparability:** After standardisation, all features are measured in units of standard deviation. A coefficient of 0.8 for `emp.var.rate` and 0.3 for `age` can be directly compared — the former has a stronger association with the log-odds of subscription. Without scaling, a large coefficient for a small-range feature (e.g., `cons.conf.idx`) would simply reflect unit conversion, not predictive importance.

3. **Regularisation penalties:** The L2 penalty in Logistic Regression adds $\lambda \sum_j w_j^2$ to the loss. It penalises large coefficient values regardless of the feature's measurement scale. Without standardisation, features with a small measurement range (e.g., `emp.var.rate` ∈ [-3, 1.4]) require *large* coefficient magnitudes to exert meaningful influence on the model output. The L2 penalty then disproportionately shrinks these large coefficients — effectively over-regularising small-scale features and under-regularising large-scale ones. Standardisation ensures the penalty is equitable across all features.

**Why not Min-Max scaling?** Min-Max normalisation maps values to [0,1] but is highly sensitive to outliers: a single extreme value in `campaign` (max = 56 contacts) compresses all other values into a narrow range. Standardisation is more robust because it is based on mean and standard deviation rather than range extrema.

In [ ]:
# Identify columns that need scaling:
# All columns that are NOT binary (i.e., not OHE dummies and not the binary indicator)
binary_cols = [c for c in X_train_enc.columns if X_train_enc[c].nunique() == 2 and X_train_enc[c].min() == 0]
scale_cols  = [c for c in X_train_enc.columns if c not in binary_cols]

print(f'Columns to scale ({len(scale_cols)}): {scale_cols}')
print(f'Binary columns skipped ({len(binary_cols)}): {binary_cols[:8]} ...' if len(binary_cols)>8 else f'Binary columns skipped: {binary_cols}')

In [ ]:
# Fit StandardScaler on TRAINING set ONLY
scaler = StandardScaler()
scaler.fit(X_train_enc[scale_cols])

# Transform all splits using the training-set scaler
X_train_scaled = X_train_enc.copy()
X_val_scaled   = X_val_enc.copy()
X_test_scaled  = X_test_enc.copy()

X_train_scaled[scale_cols] = scaler.transform(X_train_enc[scale_cols])
X_val_scaled[scale_cols]   = scaler.transform(X_val_enc[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test_enc[scale_cols])

# Verify scaling on training set
means  = X_train_scaled[scale_cols].mean().round(6)
stdevs = X_train_scaled[scale_cols].std().round(4)
print('Training set after scaling — means (should be ≈0):')
print(means.values)
print('\nTraining set after scaling — std devs (should be ≈1):')
print(stdevs.values)
print('\n✓ StandardScaler fitted on training set and applied to all splits.')

---
## Step 8 – Feature Selection
*Lecture 5 – Feature Selection | Lecture 6 – Linear Models | Lecture 9 – Pipeline Discipline*

### Why Feature Selection Must Use Training Data Only

Variance thresholds and correlations are computed on the training set only. Using full-dataset statistics would constitute selection leakage, as the retained feature space would be implicitly tuned to held-out data.

### 1. Low-Variance Features

After standardisation, all continuous features have variance ≈ 1. A feature with variance < 0.01 on the training set therefore varies by less than 1/100th of a standard deviation — it is essentially constant and contributes no discriminative information to any model, including Logistic Regression. The threshold of 0.01 is conservative: it ensures we only remove features that are near-identical across almost all training observations, minimising the risk of discarding weakly informative features.

In [ ]:
# Compute variance on training set
train_variance = X_train_scaled.var()

# Threshold: remove features whose variance < 0.01 (after standardisation, continuous
# features have variance ≈1; near-zero variance suggests near-constant columns)
VAR_THRESHOLD = 0.01
low_var_cols = train_variance[train_variance < VAR_THRESHOLD].index.tolist()
print(f'Low-variance features (variance < {VAR_THRESHOLD}) computed on training set:')
if low_var_cols:
    print(pd.DataFrame({'Variance': train_variance[low_var_cols].round(6)}))
else:
    print('  None found.')
print(f'\nTotal variance stats (training):')
print(train_variance.sort_values().head(10).round(5))

In [ ]:
# ── Variance filter: identify columns to drop (training set only) ─────────────
train_variance = X_train_scaled.var()
VAR_THRESHOLD  = 0.01

low_var_cols = train_variance[train_variance < VAR_THRESHOLD].index.tolist()

# Also check default OHE columns: EDA flagged 'default_yes' as near-constant
default_cols = [c for c in X_train_scaled.columns if 'default' in c]
for c in default_cols:
    if X_train_scaled[c].var() < VAR_THRESHOLD and c not in low_var_cols:
        low_var_cols.append(c)

print(f'Low-variance columns to drop (variance < {VAR_THRESHOLD} on training set):')
if low_var_cols:
    print(train_variance[low_var_cols].round(6).to_string())
else:
    print('  None found.')

In [ ]:
# ── Apply variance-based drop to all splits ───────────────────────────────────
if low_var_cols:
    X_train_scaled.drop(columns=low_var_cols, inplace=True)
    X_val_scaled.drop(columns=low_var_cols, inplace=True)
    X_test_scaled.drop(columns=low_var_cols, inplace=True)
    print(f'✓ Dropped {len(low_var_cols)} low-variance columns: {low_var_cols}')
else:
    print('No low-variance columns to drop.')
print(f'Shape after variance filter: {X_train_scaled.shape}')

### 2. Highly Correlated Features

The following section computes pairwise absolute correlations among continuous numerical features on the **training set only**, then removes features above the collinearity threshold. Removing features based on full-dataset correlations would constitute selection leakage (see introduction above).

### 2. Highly Correlated Features

In [ ]:
# Compute correlation matrix on TRAINING SET only
# Focus on continuous numerical features (not binary OHE columns)
cont_cols_for_corr = [c for c in scale_cols if c in X_train_scaled.columns]
corr_matrix = X_train_scaled[cont_cols_for_corr].corr().abs()

# Visualise heatmap
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=0, vmax=1, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Absolute Pairwise Correlations — Numerical Features (Training Set)',
             fontweight='bold', fontsize=12, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated pairs (threshold = 0.90)
CORR_THRESHOLD = 0.90
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = [
    (col, row, upper_tri.loc[row, col])
    for col in upper_tri.columns
    for row in upper_tri.index
    if pd.notna(upper_tri.loc[row, col]) and upper_tri.loc[row, col] >= CORR_THRESHOLD
]

print(f'Feature pairs with |correlation| ≥ {CORR_THRESHOLD}:')
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -x[2]):
    print(f'  {f1:<20} ↔  {f2:<20}: r = {r:.3f}')

### Correlation-Based Feature Removal Decision

The three macroeconomic features — `emp.var.rate`, `euribor3m`, and `nr.employed` — all capture the same underlying economic cycle and are correlated at |r| ≥ 0.94. For Logistic Regression this presents two concrete problems:

- **Multicollinearity inflates coefficient variance:** When two features carry nearly identical information, the model cannot reliably apportion predictive credit between them. Their coefficients become highly sensitive to small changes in the training data — a symptom of the *variance* component of the bias–variance tradeoff (Lecture 6).
- **L2 regularisation is less effective:** The penalty shrinks the sum of squared coefficients. When two features are near-duplicates, the optimiser can split their coefficients arbitrarily while incurring the same total penalty — the regulariser fails to enforce the intended shrinkage.

**Decision:** Remove `euribor3m` and `nr.employed`, retaining `emp.var.rate`. The rationale is semantic: the employment variation rate is a direct, interpretable measure of the economic cycle that is also directly referenced in central bank and labour market analysis. `euribor3m` and `nr.employed` are more indirect or noisy proxies of the same signal. Retaining the most semantically transparent variable preserves interpretability of the final model's macroeconomic coefficient.

**Threshold justification:** |r| ≥ 0.90 is a widely used threshold in linear modelling for identifying problematic collinearity. Below this level, two features may share variance but still contribute independent predictive information that a linear model can exploit through separate coefficients.

In [ ]:
# Remove highly correlated features
corr_drop = ['euribor3m', 'nr.employed']
# Only drop if they still exist (they might have been dropped by variance filter)
corr_drop = [c for c in corr_drop if c in X_train_scaled.columns]

X_train_scaled.drop(columns=corr_drop, inplace=True)
X_val_scaled.drop(columns=corr_drop, inplace=True)
X_test_scaled.drop(columns=corr_drop, inplace=True)

print(f'✓ Dropped correlated features: {corr_drop}')
print(f'Final feature set size: {X_train_scaled.shape[1]} features')
print(f'Final training shape:   {X_train_scaled.shape}')

**Summary:** Both filters above were computed exclusively on `X_train_scaled`. The identical column lists were then applied to `X_val_scaled` and `X_test_scaled` via `drop()` — the held-out sets inform neither the variance calculation nor the correlation matrix. This is the correct application of the pipeline discipline principle discussed at the start of this section.

---
## Step 9 – Addressing Class Imbalance
*Lecture 3 – Class Imbalance | Lecture 4 – Evaluation Metrics | Lecture 9 – Pipeline Discipline*

### Class Distribution in the Training Set

In [ ]:
print('Training set class distribution (before resampling):')
vc = y_train.value_counts()
print(pd.DataFrame({'Count': vc, 'Percentage (%)': (vc/len(y_train)*100).round(1)}))
print(f'\nImbalance ratio: {vc[0]/vc[1]:.1f}:1 (majority:minority)')

### Why Imbalance Is a Concern

An ~8:1 imbalance ratio is a **significant concern** for this task:

1. **Model bias toward the majority class:** Logistic Regression minimises the total log-loss. When 89% of training samples are negative, predicting `no` for every observation incurs very low loss. The model can therefore achieve a near-minimal training loss without learning anything about the minority class. This produces high accuracy but near-zero recall on the positive class — precisely the opposite of what the bank needs.

2. **Accuracy is a misleading metric:** The **Zero-Rule baseline** — a trivial classifier that always predicts the majority class — achieves ~89% accuracy with zero predictive content:

```python
zero_rule_acc = y_train.value_counts(normalize=True).max()
# → 0.890
```

Any model whose accuracy is close to this value has learned nothing useful about the minority class. Precision and recall are required to expose this.

### Resampling Strategy: Random Oversampling

**Method:** Randomly duplicate samples from the minority class until the two classes are balanced in the training set.

**Why random oversampling (not SMOTE)?** SMOTE generates synthetic minority samples by linear interpolation between neighbouring observations in feature space. After OHE, the feature space contains many binary dimensions where linear interpolation yields non-binary fractional values — for example, 0.3 for a column that encodes `contact_telephone` (which can only be 0 or 1). A synthetic sample with `contact_telephone = 0.3` corresponds to no real-world observation: it is neither cellular nor telephone contact. This violates SMOTE's core assumption that interpolated points are plausible, and can introduce unrealistic training samples that degrade generalisation. Random oversampling replicates genuine minority-class observations and is therefore always valid regardless of feature type.

**Why not undersample?** The training set contains approximately 317 minority-class observations. Reducing the majority class to match would leave a training set of ~634 rows total — too small for stable parameter estimation with ~41 features, and insufficient for the L2 regulariser to function correctly.

### Stage in the Pipeline

Resampling must occur **after** encoding and scaling (to operate in the final feature space) and **on the training set only**. Validation and test sets must preserve the original class distribution to simulate the real deployment scenario and produce unbiased evaluation metrics.

### What Would Happen If Resampled Before Splitting

If oversampling were performed before splitting, duplicated observations could appear in both training and validation sets, causing evaluation leakage and inflated performance.

In [ ]:
# ── Random oversampling on training set ONLY ──────────────────────────────────
# Combine features and labels for resampling
train_combined = X_train_scaled.copy()
train_combined['y'] = y_train.values

train_majority = train_combined[train_combined['y'] == 0]
train_minority = train_combined[train_combined['y'] == 1]

# Oversample minority class to match majority class size
train_minority_upsampled = resample(
    train_minority,
    replace=True,
    n_samples=len(train_majority),
    random_state=RANDOM_STATE
)

train_resampled = pd.concat([train_majority, train_minority_upsampled])
train_resampled = train_resampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_train_res = train_resampled.drop(columns=['y'])
y_train_res = train_resampled['y']

print('After oversampling:')
vc2 = y_train_res.value_counts()
print(pd.DataFrame({'Count': vc2, 'Percentage (%)': (vc2/len(y_train_res)*100).round(1)}))
print(f'\nTraining set size: {len(X_train_res):,} rows (was {len(X_train_scaled):,})')
print(f'Validation/test sets untouched: {len(X_val_scaled):,} / {len(X_test_scaled):,} rows')

### Effect of Imbalance on Evaluation Metrics

| Metric | Effect of Imbalance |
|--------|---------------------|
| **Accuracy** | Dominated by the majority class. A classifier that always predicts `no` achieves ~89% accuracy — a score that conveys no information about minority-class detection. |
| **Precision** | A conservative model that rarely predicts positive can appear highly precise (few false positives) while missing most true subscribers. High precision without high recall is not useful for this task. |
| **Recall** | A majority-biased model will have near-zero recall for the positive class — it misses most potential subscribers, which is the primary business cost. |

For this use case, **recall for the positive class is the most business-critical metric**: the cost of missing a subscriber (lost revenue) substantially outweighs the cost of a wasted call. Accuracy alone would give a misleading picture of whether the pipeline is working.

---
## Step 10 – Training a Logistic Regression Model
*Lecture 6 – Logistic Regression | Lecture 9–11 – Model Evaluation and Metrics*

### Why Logistic Regression as a Sanity Check?

Logistic Regression is appropriate here not because it is expected to be the best model, but because it is the right tool for verifying that the data preparation pipeline is coherent. It is a **convex, deterministic model**: given the same data and hyperparameters, it always converges to the same solution. If the pipeline is correct — encoding is consistent, scaling is applied before training, resampling is training-only — the model should converge cleanly and produce finite, interpretable coefficients. Anomalous behaviour (non-convergence, degenerate coefficients, near-zero recall on both classes simultaneously) signals a pipeline error rather than a modelling limitation. This diagnostic role is exactly what the assignment means by "consistency check."

In [ ]:
# ── Train Logistic Regression on the resampled training set ───────────────────
lr = LogisticRegression(
    penalty='l2',        # L2 regularisation — standard for Logistic Regression
    C=1.0,               # Inverse regularisation strength; C=1 is sklearn default
    solver='lbfgs',      # Quasi-Newton solver, efficient for small/medium datasets
    max_iter=1000,       # Sufficient iterations for convergence
    random_state=RANDOM_STATE
)

lr.fit(X_train_res, y_train_res)
print('✓ Logistic Regression trained.')

# ── Generate predictions on the validation set ────────────────────────────────
y_pred_val = lr.predict(X_val_scaled)

In [ ]:
# ── Zero-Rule Baseline (fit on ORIGINAL training distribution) ──

dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)

dummy.fit(X_train_scaled, y_train)

y_dummy_val = dummy.predict(X_val_scaled)

dummy_acc  = accuracy_score(y_val, y_dummy_val)
dummy_prec = precision_score(y_val, y_dummy_val, zero_division=0)
dummy_rec  = recall_score(y_val, y_dummy_val, zero_division=0)

zero_rule = dummy_acc

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────────
import pandas as pd
comparison = pd.DataFrame({
    'Model'    : ['Zero-Rule (always predict majority)', 'Logistic Regression'],
    'Accuracy' : [dummy_acc,  acc],
    'Precision': [dummy_prec, prec],
    'Recall'   : [dummy_rec,  rec],
})
print('\n── Validation Set Performance ──────────────────────────────────────')
print(comparison.round(4).to_string(index=False))
print('────────────────────────────────────────────────────────────────────')
print(f'\nKey insight: Zero-Rule achieves {dummy_acc:.1%} accuracy with {dummy_rec:.0%} recall.')
print(f'Logistic Regression achieves {acc:.1%} accuracy with {rec:.1%} recall.')
print(f'Lower accuracy ({acc:.1%} vs {dummy_acc:.1%}) is expected after oversampling;')
print(f'the {rec:.1%} recall gain is the meaningful result.')

In [ ]:
# ── Full classification report ────────────────────────────────────────────────
print('Full Classification Report:')
print(classification_report(y_val, y_pred_val, target_names=['No Subscribe (0)', 'Subscribe (1)']))

In [ ]:
# ── Professional Confusion Matrix Visualisation ───────────────────────────────
cm = confusion_matrix(y_val, y_pred_val)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Left panel: Styled Confusion Matrix ──────────────────────────────────────
ax = axes[0]
# Custom colour grid: correct predictions in green shades, errors in red shades
cell_colours = np.array([['#C8E6C9', '#FFCDD2'],
                          ['#FFCDD2', '#C8E6C9']])
for i in range(2):
    for j in range(2):
        ax.add_patch(plt.Rectangle((j, 1-i), 1, 1, color=cell_colours[i, j]))
        ax.text(j + 0.5, 1.5 - i, str(cm[i, j]),
                ha='center', va='center', fontsize=26, fontweight='bold',
                color='#1B5E20' if i == j else '#B71C1C')
        label = ['True Neg', 'False Pos', 'False Neg', 'True Pos'][(i*2)+j]
        ax.text(j + 0.5, 1.15 - i, label,
                ha='center', va='center', fontsize=9, color='#424242')

ax.set_xlim(0, 2); ax.set_ylim(0, 2)
ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(['Predicted: No', 'Predicted: Yes'], fontsize=11)
ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(['Actual: Yes', 'Actual: No'], fontsize=11, rotation=90, va='center')
ax.set_title('Confusion Matrix — Validation Set', fontweight='bold', fontsize=13, pad=12)
ax.tick_params(length=0)
for spine in ax.spines.values(): spine.set_visible(False)

# ── Right panel: Metric Comparison Bar Chart ──────────────────────────────────
ax2 = axes[1]
metrics_names  = ['Accuracy', 'Precision', 'Recall', 'Zero-Rule\nBaseline']
metrics_values = [acc, prec, rec, zero_rule]
bar_colours    = ['#1565C0', '#6A1B9A', '#E65100', '#9E9E9E']

bars = ax2.bar(metrics_names, metrics_values, color=bar_colours, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, metrics_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax2.set_ylim(0, 1.15)
ax2.set_title('Model Performance vs. Zero-Rule Baseline', fontweight='bold', fontsize=13)
ax2.set_ylabel('Score')
ax2.axhline(y=zero_rule, color='#9E9E9E', linestyle='--', linewidth=1.2, alpha=0.7)

plt.suptitle('Logistic Regression — Validation Set Results',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Interpretation of Results

**Accuracy vs. Zero-Rule Baseline:**
The Zero-Rule baseline achieves 89.0% accuracy with 0% recall — it predicts `no` for every observation and learns nothing about subscribers. Our model's accuracy is 82.7%, which is lower. This is the **expected and correct** consequence of oversampling: the model now actively predicts the minority class rather than ignoring it, which increases false positives and reduces overall accuracy while dramatically increasing recall.

**Recall for the positive class:**
The model achieves 63.2% recall on the validation set, compared to 0% for the Zero-Rule baseline. This means the model identifies approximately 63 out of every 100 actual subscribers — a substantial improvement in what the bank actually cares about.

**Precision:**
Precision for the positive class is 34.4%. This reflects the precision–recall tradeoff introduced by oversampling: more positive predictions means more false positives. For a telemarketing campaign, this tradeoff is operationally acceptable — the cost of a wasted call is substantially lower than the cost of a missed subscriber.

**If validation performance were unexpectedly poor** (e.g., recall near zero despite oversampling), that would signal a pipeline error — most likely leakage or incorrect ordering of transformations. In that case, the correct response is to restart from the Data Splitting step and verify each transformation's fit/transform separation.

In [ ]:
# ── Top feature coefficients ──────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature': X_train_res.columns,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(15).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E53935' if c > 0 else '#1E88E5' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'][::-1], coef_df['Coefficient'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient (log-odds)')
ax.set_title('Top 15 Feature Coefficients (Standardised)', fontweight='bold', fontsize=13)
red_p  = mpatches.Patch(color='#E53935', label='Positive → increases P(subscribe)')
blue_p = mpatches.Patch(color='#1E88E5', label='Negative → decreases P(subscribe)')
ax.legend(handles=[red_p, blue_p], fontsize=9)
plt.tight_layout()
plt.show()

print('Because features are standardised, coefficient magnitudes are directly comparable.')
print('Larger |coefficient| → stronger association with subscription outcome.')

**Pipeline sanity check:** All 41 coefficients in the plot above are finite and of interpretable magnitude — no extreme values or NaNs that would indicate numerical instability. The model does not trivially predict one class. These are positive signals that the pipeline is correctly ordered and free of obvious leakage.

---
## Summary: End-to-End Pipeline

| Step | Operation | Fitted on | Applied to |
|:----:|-----------|-----------|------------|
| 1 | Identify target (`y`) | — | — |
| 2 | Data loading & EDA | Full dataset (read-only) | — |
| 3 | Drop `duration`; create `pdays_contacted`; encode target | No params (stateless) | All data |
| 4 | Stratified train/val/test split (70/15/15) | — | All data |
| 5 | Median imputation for `pdays` | **Train only** | All splits |
| 6a | Ordinal encoding for `education` | **Train vocabulary** | All splits |
| 6b | One-hot encoding for nominal variables | **Train vocabulary** | All splits |
| 7 | `StandardScaler` (`.fit` on train, `.transform` on all) | **Train only** | All splits |
| 8a | Variance filter (< 0.01) | **Train only** | All splits |
| 8b | Correlation filter (|r| > 0.90) | **Train only** | All splits |
| 9 | Random oversampling of minority class | **Train only** | Train only |
| 10 | Zero-Rule baseline (DummyClassifier) | Resampled train | Validation |
| 10 | Logistic Regression training | Resampled train | Train |
| 10 | Evaluation (accuracy, precision, recall) | — | Validation set |

Every stateful operation (Steps 5–9) is fitted exclusively on the training set. `StandardScaler` uses sklearn's `.fit()`/`.transform()` API; categorical encoders are applied by reindexing to the training column schema with unseen categories filled as zero. Validation and test sets are only ever transformed — never fit upon. The test set is not evaluated in this notebook; it is reserved for a final, fully unseen assessment after all pipeline decisions have been locked.